# Exploratory Data Analysis

This notebook examines the cleaned VNAT monthly segment dataset. The narrative emphasizes seasonality, shock sensitivity, and market heterogeneity.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import CLEAN_SEGMENTS, FIGURES, TABLES, TARGET_COLUMNS, SEGMENT_COLUMNS, SERIES_COLORS, SERIES_LABELS
from src.plotting import annotate_events, save_figure, set_academic_style

set_academic_style()
df = pd.read_csv(CLEAN_SEGMENTS, parse_dates=["date"]).set_index("date").sort_index()
df.index.freq = "MS"

## Total Arrivals

Observation: total arrivals rise before the pandemic, collapse during COVID-19, and recover after reopening. Statistical implication: the mean process is structurally unstable. Tourism implication: forecasts should be interpreted with explicit shock uncertainty.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df["international_arrivals"], color=SERIES_COLORS["international_arrivals"])
annotate_events(ax, ["COVID-19 outbreak", "Vietnam border reopening", "Tourism recovery period"])
ax.set_title("Structural Shocks in International Tourist Arrivals")
ax.set_ylabel("Monthly arrivals")
save_figure(fig, FIGURES / "01_total_arrivals_timeseries.png")
plt.show()

## Segment Arrivals

Observation: Asia contributes the largest volume, while other regions have smaller but distinct trajectories. Statistical implication: aggregate arrivals mask segment-level heterogeneity. Tourism implication: recovery strategies should not assume uniform market behavior.

In [ ]:
fig, ax = plt.subplots()
for col in SEGMENT_COLUMNS:
    ax.plot(df.index, df[col], label=SERIES_LABELS[col], color=SERIES_COLORS[col])
ax.set_title("Regional Composition of Vietnam International Arrivals")
ax.set_ylabel("Monthly arrivals")
ax.legend(ncol=3, fontsize=8)
save_figure(fig, FIGURES / "02_segment_arrivals.png")
plt.show()

## Segment Shares

Observation: source-market shares vary through time, especially during reopening. Statistical implication: composition shifts are relevant for forecast interpretation. Tourism implication: aviation and marketing resources should be aligned with changing source-region shares.

In [ ]:
shares = df[SEGMENT_COLUMNS].div(df["international_arrivals"], axis=0) * 100
fig, ax = plt.subplots()
bottom = np.zeros(len(shares))
for col in SEGMENT_COLUMNS:
    values = shares[col].to_numpy()
    ax.fill_between(shares.index, bottom, bottom + values, color=SERIES_COLORS[col], alpha=0.75, label=SERIES_LABELS[col])
    bottom += values
ax.set_ylim(0, 100)
ax.set_title("Changing Market Shares by Source Region")
ax.set_ylabel("Share of total arrivals (%)")
ax.legend(ncol=3, fontsize=8)
save_figure(fig, FIGURES / "03_segment_share_over_time.png")
plt.show()

## Monthly Seasonality

Observation: arrivals follow a recurring monthly profile. Statistical implication: seasonal terms are necessary in forecasting models. Tourism implication: capacity planning should account for predictable intra-year peaks.

In [ ]:
monthly = df.assign(month=df.index.month).groupby("month")["international_arrivals"]
med = monthly.median()
q1 = monthly.quantile(0.25)
q3 = monthly.quantile(0.75)
fig, ax = plt.subplots()
ax.plot(med.index, med.values, color=SERIES_COLORS["international_arrivals"])
ax.fill_between(med.index, q1.values, q3.values, color=SERIES_COLORS["international_arrivals"], alpha=0.18)
ax.set_xticks(range(1, 13))
ax.set_title("Seasonal Dynamics of International Tourist Arrivals")
ax.set_ylabel("Median arrivals")
save_figure(fig, FIGURES / "04_monthly_seasonality.png")
plt.show()

## Yearly Trend

Observation: the annual series separates the pre-pandemic expansion, pandemic trough, and recovery. Statistical implication: the trend component is interrupted by a structural break. Tourism implication: long-run planning should distinguish trend recovery from temporary rebound effects.

In [ ]:
annual = df["international_arrivals"].resample("YS").sum()
fig, ax = plt.subplots()
ax.plot(annual.index.year, annual.values, marker="o", color=SERIES_COLORS["international_arrivals"])
ax.set_title("Annual Trend and Post-Pandemic Recovery Path")
ax.set_ylabel("Annual arrivals")
save_figure(fig, FIGURES / "05_yearly_trend.png")
plt.show()

## Growth Rates and COVID Shock

Observation: year-on-year growth is extremely volatile around the pandemic and reopening. Statistical implication: variance is time-varying after large shocks. Tourism implication: policy decisions should avoid treating rebound growth as stable baseline demand.

In [ ]:
growth = df["international_arrivals"].pct_change(12) * 100
fig, ax = plt.subplots()
ax.axhline(0, color="#777777", linewidth=0.8)
ax.plot(growth.index, growth, color=SERIES_COLORS["international_arrivals"])
annotate_events(ax, ["COVID-19 outbreak", "Vietnam border reopening"])
ax.set_title("Year-on-Year Growth and Shock Transmission")
ax.set_ylabel("Year-on-year growth (%)")
save_figure(fig, FIGURES / "06_growth_rate_analysis.png")
plt.show()

## Correlation Structure

Observation: segment growth rates are positively correlated but not identical. Statistical implication: common shocks dominate, while segment-specific dynamics remain. Tourism implication: diversification across source markets may reduce exposure to region-specific demand shocks.

In [ ]:
corr = df[TARGET_COLUMNS].pct_change().corr()
fig, ax = plt.subplots(figsize=(6.8, 5.6))
im = ax.imshow(corr, cmap="Greys", vmin=-1, vmax=1)
labels = [SERIES_LABELS[c] for c in TARGET_COLUMNS]
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Correlation Structure of Monthly Arrival Growth")
save_figure(fig, FIGURES / "07_correlation_heatmap.png")
plt.show()